# 🤖 Notebook 02 — Recommendation Modeling
## TourMate AI · Tourism Recommendation System

> **Objective:** Build, train, and evaluate 7 complementary recommendation approaches.  
> Cover popularity-based, content-based, budget-aware, classification, regression, clustering, and hybrid methods.

| Approach | Algorithm | Use Case |
|---|---|---|
| Popularity Baseline | Weighted score | Top-N by volume & rating |
| Content-Based | TF-IDF + Cosine | Similar destination retrieval |
| Budget-Aware | Custom scoring | Price-filtered recommendations |
| Classification | Random Forest | Predict recommendation level |
| Regression | Gradient Boosting | Predict attraction rating |
| Clustering | KMeans + PCA | Traveler segment profiling |
| **Hybrid** | Weighted ensemble | **Final production model** |

---

In [ ]:
# ─── 0. SETUP ──────────────────────────────────────────────────────────────────
import sys, os, json, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, GradientBoostingRegressor,
                               RandomForestRegressor, ExtraTreesRegressor)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              mean_squared_error, r2_score, mean_absolute_error,
                              roc_auc_score, silhouette_score)
from sklearn.pipeline import Pipeline
import scipy.stats as stats

CHART_COLORS = ['#0ea5e9','#52c27a','#d4a373','#f97316','#8b5cf6','#ec4899','#14b8a6','#ef4444']
PALETTE = {'sky':'#0ea5e9','sky_d':'#0284c7','sky_l':'#e0f2fe','mint':'#52c27a','sand':'#d4a373'}
plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})

print("✓ All imports successful")

# Try optional boosting libraries
try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGB = True; print("✓ XGBoost available")
except ImportError:
    HAS_XGB = False; print("⚠ XGBoost not installed (optional)")

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    HAS_LGB = True; print("✓ LightGBM available")
except ImportError:
    HAS_LGB = False; print("⚠ LightGBM not installed (optional)")


In [ ]:
# ─── 1. LOAD & PREPROCESS ──────────────────────────────────────────────────────
from src.data_preprocessing import load_dataset, clean_dataset, build_attraction_catalog

df = load_dataset('../data/tourism_recommendation_dataset_en.csv')
df = clean_dataset(df)
catalog = build_attraction_catalog(df)

print(f"Raw dataset   : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Attraction catalog: {len(catalog)} unique attractions")
print(f"\nCatalog columns: {catalog.columns.tolist()}")
catalog.head(3)


## 1. Popularity-Based Recommender (Baseline)

In [ ]:
# ─── 1.1 POPULARITY SCORE CONSTRUCTION ────────────────────────────────────────
# Weighted popularity formula — explained component by component

# Component weights (justify each)
W_FREQ   = 0.40  # Visit frequency — most direct demand signal
W_RATING = 0.30  # Avg rating — quality filter
W_HREC   = 0.20  # Highly Recommend % — intent-to-revisit proxy
W_VSAT   = 0.10  # Very Satisfied % — outcome quality

cat_stats = catalog.copy()

def min_max(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

cat_stats['pop_score'] = (
    W_FREQ   * min_max(cat_stats['visit_count']) +
    W_RATING * min_max(cat_stats['avg_rating']) +
    W_HREC   * cat_stats['highly_recommended_pct'] +
    W_VSAT   * cat_stats['very_satisfied_pct']
)

print("Popularity Score Formula:")
print(f"  pop_score = {W_FREQ}×norm(visit_count)")
print(f"            + {W_RATING}×norm(avg_rating)")
print(f"            + {W_HREC}×highly_recommended_pct")
print(f"            + {W_VSAT}×very_satisfied_pct")
print()
print("Top 10 by Popularity Score:")
top10 = cat_stats.nlargest(10,'pop_score')[
    ['attraction_name','province','category','avg_rating','visit_count',
     'pop_score','budget_level']].reset_index(drop=True)
top10.index += 1
display(top10.style
    .background_gradient(subset=['pop_score'], cmap='Blues')
    .format({'avg_rating':'{:.3f}','pop_score':'{:.4f}','visit_count':'{:,}'})
    .set_properties(**{'font-size':'11.5px'}))


In [ ]:
# ─── 1.2 POPULARITY RECOMMENDER — FILTERED USE CASES ──────────────────────────
from src.recommender import PopularityRecommender

pop_rec = PopularityRecommender()
pop_rec.fit(catalog)

test_cases = [
    ('Natural Scenery', None, 'Spring'),
    ('Historical Culture', 'Henan', None),
    ('Theme Park', None, 'Summer'),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, (cat, prov, season) in zip(axes, test_cases):
    result = pop_rec.recommend(category=cat, province=prov, season=season, n=8)
    label = f"{cat[:12]}\n{prov or 'Any Province'} · {season or 'Any Season'}"
    ax.barh(range(len(result)), result['popularity_score'].values[::-1],
            color=CHART_COLORS[:len(result)], edgecolor='white')
    ax.set_yticks(range(len(result)))
    ax.set_yticklabels(result['attraction_name'].values[::-1], fontsize=8)
    ax.set_xlabel('Popularity Score')
    ax.set_title(f'Top Recommendations\n{label}', fontsize=11, fontweight='bold')
    ax.set_xlim(0, 1.15)
    for i, (v, name) in enumerate(zip(result['popularity_score'].values[::-1],
                                       result['attraction_name'].values[::-1])):
        ax.text(v+0.02, i, f'{v:.3f}', va='center', fontsize=8)

plt.suptitle('Popularity-Based Recommender — Three Use Cases', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Content-Based Recommender (TF-IDF + Cosine Similarity)

In [ ]:
# ─── 2.1 TF-IDF FEATURE ENGINEERING ───────────────────────────────────────────
from src.recommender import ContentBasedRecommender

# Inspect what combined_features looks like
print("Sample combined_features strings:")
for _, row in catalog.head(3).iterrows():
    print(f"  [{row['attraction_name'][:25]}]")
    print(f"   → '{row['combined_features']}'")
    print()

# Build content recommender
content_rec = ContentBasedRecommender()
content_rec.fit(catalog)

print(f"TF-IDF matrix shape      : {content_rec.tfidf_matrix.shape}")
print(f"Vocabulary size          : {len(content_rec.tfidf.vocabulary_):,} terms")
print(f"NearestNeighbors fitted  : {content_rec.nn_model.n_samples_fit_} samples")
print(f"Similarity metric        : cosine")
print(f"Max ngram range          : (1, 2) — unigrams + bigrams")


In [ ]:
# ─── 2.2 TF-IDF TOP TERMS — PER CATEGORY ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(22, 9))
axes = axes.flat

top_cats = catalog['category'].value_counts().head(8).index.tolist()
for ax, cat in zip(axes, top_cats):
    cat_corpus = catalog[catalog['category']==cat]['combined_features'].fillna('').tolist()
    if not cat_corpus:
        ax.axis('off'); continue
    tfidf_cat = TfidfVectorizer(max_features=200, ngram_range=(1,2), stop_words=None)
    try:
        mat = tfidf_cat.fit_transform(cat_corpus)
        mean_scores = np.asarray(mat.mean(axis=0)).flatten()
        top_idx = mean_scores.argsort()[-10:][::-1]
        terms   = np.array(tfidf_cat.get_feature_names_out())[top_idx]
        scores  = mean_scores[top_idx]
        ax.barh(range(10), scores[::-1],
                color=plt.cm.Blues_r(np.linspace(0.3,0.9,10))[::-1])
        ax.set_yticks(range(10))
        ax.set_yticklabels(terms[::-1], fontsize=8)
        ax.set_title(f'{cat[:20]}', fontsize=10, fontweight='bold')
        ax.set_xlabel('Mean TF-IDF', fontsize=8)
    except Exception:
        ax.axis('off')

plt.suptitle('Top TF-IDF Terms per Category (Mean Scores)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─── 2.3 SIMILARITY MATRIX VISUALISATION ──────────────────────────────────────
# Select a representative sample of 40 attractions (8 categories × 5 each)
sample_names = []
for cat in catalog['category'].value_counts().head(8).index:
    sub = catalog[catalog['category']==cat].nlargest(5,'popularity_score')
    sample_names.extend(sub['attraction_name'].tolist())
sample_names = sample_names[:40]

sample_idx = [catalog[catalog['attraction_name']==n].index[0]
              for n in sample_names if not catalog[catalog['attraction_name']==n].empty]
sample_mat  = content_rec.tfidf_matrix[sample_idx]
sim_matrix  = cosine_similarity(sample_mat)
sample_labels = [catalog.loc[i,'attraction_name'][:18] for i in sample_idx]

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Cosine Similarity', shrink=0.7)
ax.set_xticks(range(len(sample_labels)))
ax.set_yticks(range(len(sample_labels)))
ax.set_xticklabels(sample_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(sample_labels, fontsize=7)
ax.set_title('Content-Based Similarity Matrix\n(Top 40 Attractions — 8 Categories)',
             fontsize=13, fontweight='bold', pad=12)

# Add category separators
cat_boundaries = []
current_cat = None
for i, idx in enumerate(sample_idx):
    cat = catalog.loc[idx,'category']
    if cat != current_cat:
        if i > 0: cat_boundaries.append(i - 0.5)
        current_cat = cat
for b in cat_boundaries:
    ax.axhline(b, color='white', lw=1.5)
    ax.axvline(b, color='white', lw=1.5)

plt.tight_layout()
plt.savefig('../static/charts/nb2_similarity_matrix.png', bbox_inches='tight', dpi=130)
plt.show()

# Print distribution of similarity scores
tri = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
print(f"Pairwise cosine similarity stats:")
print(f"  Mean   : {tri.mean():.4f}")
print(f"  Std    : {tri.std():.4f}")
print(f"  Min    : {tri.min():.4f}")
print(f"  Max    : {tri.max():.4f}")
print(f"  >0.50  : {(tri > 0.50).sum()} pairs ({(tri>0.50).mean()*100:.1f}%)")
print(f"  >0.80  : {(tri > 0.80).sum()} pairs ({(tri>0.80).mean()*100:.1f}%)")


In [ ]:
# ─── 2.4 SIMILARITY QUERIES — 4 USE CASES ────────────────────────────────────
test_queries = [
    {'type': 'destination', 'name': 'Huang Shan',
     'label': 'Find attractions similar to Huang Shan (scenic mountain)'},
    {'type': 'destination', 'name': 'Xi Hu',
     'label': 'Find attractions similar to Xi Hu (West Lake, urban scenery)'},
    {'type': 'query', 'q': 'ancient town history culture spring',
     'label': 'Free-text: ancient cultural towns in spring'},
    {'type': 'query', 'q': 'theme park family summer budget',
     'label': 'Free-text: family theme parks, summer, budget-friendly'},
]

for tq in test_queries:
    print("─" * 70)
    print(f"Query: {tq['label']}")
    if tq['type'] == 'destination':
        result = content_rec.get_similar(tq['name'], n=6)
    else:
        result = content_rec.search_by_query(tq['q'], n=6)

    if not result.empty:
        display(result[['attraction_name','province','category','avg_rating',
                         'budget_level','similarity_score']]
            .style.background_gradient(subset=['similarity_score'], cmap='Blues')
            .format({'avg_rating':'{:.3f}','similarity_score':'{:.4f}'})
            .set_properties(**{'font-size':'11px'}))
    print()


## 3. Budget-Aware Preference Recommender

In [ ]:
# ─── 3.1 CUSTOM SCORING FORMULA ───────────────────────────────────────────────
# Full custom scoring with explicit formula components + penalties
def budget_aware_recommend(catalog, content_rec, df_raw,
                            category=None, season=None, budget_level=None,
                            province=None, min_rating=3.5, n=10):
    """
    Custom Budget-Aware Recommender with explicit scoring formula:

    final_score = w_sim   × similarity_score
                + w_pop   × popularity_score (normalized)
                + w_rate  × rating_score (normalized)
                + w_cat   × category_match_bonus
                + w_season× season_match_bonus
                - penalty_budget  × budget_mismatch (0 or 1)
                - penalty_low_rate× max(0, 4.0 - avg_rating) / 4.0
    """
    WEIGHTS = dict(sim=0.28, pop=0.22, rate=0.20, cat=0.12,
                   season=0.10, budget_pen=0.08, low_rate_pen=0.05)

    df = catalog.copy()

    # Step 1: content similarity
    query_parts = []
    if category: query_parts.append(category)
    if province: query_parts.append(province)
    if season:   query_parts.append(season)
    if budget_level: query_parts.append(budget_level)
    query = ' '.join(query_parts) if query_parts else 'popular tourism china'

    sim_result = content_rec.search_by_query(query, n=len(df))
    sim_lookup = dict(zip(sim_result['attraction_name'],
                          sim_result.get('similarity_score', pd.Series([0.5]*len(sim_result)))))
    df['sim_score'] = df['attraction_name'].map(sim_lookup).fillna(0.3)

    # Step 2: normalize components
    def norm(s): return (s - s.min()) / (s.max() - s.min() + 1e-9)
    df['pop_norm']    = norm(df['popularity_score'])
    df['rating_norm'] = norm(df['avg_rating'])

    # Step 3: categorical bonuses
    df['cat_bonus']    = (df['category'].str.lower() == (category or '').lower()).astype(float) if category else 0.5
    df['season_bonus'] = (df['best_season'].str.lower() == (season or '').lower()).astype(float) if season else 0.5

    # Step 4: budget mismatch penalty
    if budget_level:
        df['budget_pen'] = (df['budget_level'].astype(str).str.lower() != budget_level.lower()).astype(float)
    else:
        df['budget_pen'] = 0.0

    # Step 5: low rating penalty
    df['low_rate_pen'] = (4.0 - df['avg_rating']).clip(lower=0) / 4.0

    # Step 6: compute final score
    w = WEIGHTS
    df['final_score'] = (
        w['sim']       * df['sim_score'] +
        w['pop']       * df['pop_norm'] +
        w['rate']      * df['rating_norm'] +
        w['cat']       * df['cat_bonus'] +
        w['season']    * df['season_bonus'] -
        w['budget_pen']    * df['budget_pen'] -
        w['low_rate_pen'] * df['low_rate_pen']
    )

    # Step 7: filter by min_rating
    df = df[df['avg_rating'] >= min_rating]
    if province:
        df_prov = df[df['province'].str.lower() == province.lower()]
        if len(df_prov) >= n // 2: df = df_prov

    result = df.nlargest(n, 'final_score').copy().reset_index(drop=True)
    result['match_pct'] = (result['final_score'] / result['final_score'].max() * 100).round(1)
    return result, WEIGHTS

# Test the formula
result_ba, weights = budget_aware_recommend(
    catalog, content_rec, df,
    category='Historical Culture', season='Autumn',
    budget_level='Mid-Range', min_rating=4.0, n=8
)

print("Budget-Aware Recommender — Historical Culture, Autumn, Mid-Range Budget:")
print(f"\nFormula weights: {weights}")
display(result_ba[['attraction_name','province','category','avg_rating',
                    'budget_level','best_season','final_score','match_pct']]
    .style.background_gradient(subset=['final_score'], cmap='Blues')
    .format({'avg_rating':'{:.3f}','final_score':'{:.4f}','match_pct':'{:.1f}%'})
    .set_properties(**{'font-size':'11px'}))


## 4. Classification Models — Predict Recommendation Level

In [ ]:
# ─── 4.1 FEATURE ENGINEERING FOR ML ──────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

df_ml = df.copy()

# Encode target
le_target = LabelEncoder()
y = le_target.fit_transform(df_ml['recommendation_level'].astype(str))
print("Target classes:", dict(zip(range(len(le_target.classes_)), le_target.classes_)))

# Encode categorical features
enc_cols = ['attraction_category','season','gender','age_group','attraction_level','province']
encoders = {}
for col in enc_cols:
    le = LabelEncoder()
    df_ml[col+'_enc'] = le.fit_transform(df_ml[col].astype(str))
    encoders[col] = le

# Feature set with engineering
feature_defs = {
    'age':                    'raw age',
    'ticket_price':           'raw ticket price',
    'visit_duration_hours':   'raw visit duration',
    'spend_amount':           'raw total spend',
    'other_spend':            'raw other spend',
    'trip_days':              'trip length',
    'is_holiday_enc':         'holiday binary',
    'is_group_tour_enc':      'group tour binary',
    'attraction_category_enc':'encoded category',
    'season_enc':             'encoded season',
    'gender_enc':             'encoded gender',
    'age_group_enc':          'encoded age group',
    'attraction_level_enc':   'encoded quality tier',
    'province_enc':           'encoded destination province',
}
FEATURE_COLS = [c for c in feature_defs if c in df_ml.columns]
X = df_ml[FEATURE_COLS].fillna(0)

print(f"\nFeature matrix: {X.shape[0]:,} samples × {X.shape[1]} features")
print("\nFeature inventory:")
for col in FEATURE_COLS:
    print(f"  {col:<28} → {feature_defs.get(col,'')}")


In [ ]:
# ─── 4.2 TRAIN/TEST SPLIT + BASELINE STATS ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

# Class distribution
print("\nClass distribution in train set:")
for cls, count in zip(le_target.classes_, np.bincount(y_train)):
    print(f"  {cls:<22} : {count:6,} ({count/len(y_train)*100:.2f}%)")

# Dummy baseline
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='stratified', random_state=42)
dummy.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, dummy.predict(X_test))
print(f"\nStratified Dummy Baseline Accuracy: {baseline_acc:.4f}")


In [ ]:
# ─── 4.3 TRAIN ALL CLASSIFIERS ────────────────────────────────────────────────
classifiers = {
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42, C=1.0),
    'Random Forest':        RandomForestClassifier(n_estimators=200, max_depth=15,
                                class_weight='balanced', random_state=42, n_jobs=-1),
    'Extra Trees':          ExtraTreesClassifier(n_estimators=200, max_depth=15,
                                class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=150, max_depth=5,
                                learning_rate=0.1, random_state=42),
}
if HAS_XGB:
    classifiers['XGBoost'] = XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.1,
                                            use_label_encoder=False, eval_metric='mlogloss',
                                            random_state=42, verbosity=0)
if HAS_LGB:
    classifiers['LightGBM'] = LGBMClassifier(n_estimators=150, num_leaves=63,
                                              class_weight='balanced', random_state=42, verbose=-1)

clf_results = {}
print(f"{'Model':<25} {'Accuracy':>10} {'F1-Weighted':>12} {'CV Mean':>10} {'CV Std':>8}")
print("─" * 70)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred  = clf.predict(X_test)
    acc     = accuracy_score(y_test, y_pred)
    f1      = f1_score(y_test, y_pred, average='weighted')
    cv_scores = cross_val_score(clf, X, y, cv=skf, scoring='f1_weighted', n_jobs=-1)
    clf_results[name] = {'model': clf, 'acc': acc, 'f1': f1,
                          'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(), 'y_pred': y_pred}
    print(f"  {name:<23} {acc:>10.4f} {f1:>12.4f} {cv_scores.mean():>10.4f} {cv_scores.std():>8.4f}")

best_clf_name = max(clf_results, key=lambda k: clf_results[k]['f1'])
best_clf = clf_results[best_clf_name]['model']
print(f"\n✓ Best Classifier: {best_clf_name} (F1={clf_results[best_clf_name]['f1']:.4f})")


In [ ]:
# ─── 4.4 BEST MODEL DEEP EVALUATION ──────────────────────────────────────────
best_y_pred = clf_results[best_clf_name]['y_pred']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Confusion matrix
cm = confusion_matrix(y_test, best_y_pred)
ConfusionMatrixDisplay(cm, display_labels=le_target.classes_).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Confusion Matrix\n{best_clf_name}', fontsize=12, fontweight='bold')

# Feature importance
if hasattr(best_clf, 'feature_importances_'):
    fi = pd.Series(best_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    axes[1].barh(range(len(fi)), fi.values[::-1],
                 color=plt.cm.Blues_r(np.linspace(0.2, 0.9, len(fi)))[::-1])
    axes[1].set_yticks(range(len(fi)))
    axes[1].set_yticklabels(fi.index[::-1], fontsize=9)
    axes[1].set_title(f'Feature Importances\n{best_clf_name}', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Importance')
    for i, v in enumerate(fi.values[::-1]):
        axes[1].text(v+0.001, i, f'{v:.4f}', va='center', fontsize=8)

# Classifier comparison bar
model_names = list(clf_results.keys())
f1_scores   = [clf_results[k]['f1'] for k in model_names]
colors_comp = [PALETTE['sand'] if k != best_clf_name else PALETTE['sky'] for k in model_names]
bars = axes[2].bar(model_names, f1_scores, color=colors_comp, edgecolor='white')
axes[2].set_ylim(0, 1.1)
for bar, v in zip(bars, f1_scores):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+0.01,
                 f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='600')
axes[2].axhline(baseline_acc, color='gray', ls='--', lw=1.5,
                label=f'Dummy baseline={baseline_acc:.3f}')
axes[2].set_title('Classifier F1-Weighted Comparison', fontsize=12, fontweight='bold')
axes[2].set_ylabel('F1-Weighted')
axes[2].set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
axes[2].legend(fontsize=9)

plt.suptitle(f'Classification Analysis — Target: recommendation_level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nDetailed Report — {best_clf_name}:")
print(classification_report(y_test, best_y_pred, target_names=le_target.classes_))


## 5. Regression Models — Predict Attraction Rating

In [ ]:
# ─── 5.1 TRAIN ALL REGRESSORS ─────────────────────────────────────────────────
y_reg = df_ml['rating'].values

regressors = {
    'Ridge Regression':       Ridge(alpha=1.0),
    'Random Forest':          RandomForestRegressor(n_estimators=200, max_depth=12,
                                  random_state=42, n_jobs=-1),
    'Extra Trees':            ExtraTreesRegressor(n_estimators=200, max_depth=12,
                                  random_state=42, n_jobs=-1),
    'Gradient Boosting':      GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                  learning_rate=0.1, subsample=0.8, random_state=42),
}
if HAS_XGB:
    regressors['XGBoost'] = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1,
                                          subsample=0.8, random_state=42, verbosity=0)
if HAS_LGB:
    regressors['LightGBM'] = LGBMRegressor(n_estimators=200, num_leaves=63,
                                            random_state=42, verbose=-1)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.20, random_state=42
)

reg_results = {}
print(f"{'Model':<25} {'R²':>8} {'RMSE':>8} {'MAE':>8} {'CV R²':>8} {'CV Std':>8}")
print("─" * 68)

from sklearn.model_selection import KFold
kf = KFold(n_splits=3, shuffle=True, random_state=42)
for name, reg in regressors.items():
    reg.fit(X_train_r, y_train_r)
    y_pred_r = reg.predict(X_test_r)
    r2   = r2_score(y_test_r, y_pred_r)
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
    mae  = mean_absolute_error(y_test_r, y_pred_r)
    cv   = cross_val_score(reg, X, y_reg, cv=kf, scoring='r2', n_jobs=-1)
    reg_results[name] = {'model': reg, 'r2': r2, 'rmse': rmse, 'mae': mae,
                          'cv_mean': cv.mean(), 'cv_std': cv.std(), 'y_pred': y_pred_r}
    print(f"  {name:<23} {r2:>8.4f} {rmse:>8.4f} {mae:>8.4f} {cv.mean():>8.4f} {cv.std():>8.4f}")

best_reg_name = max(reg_results, key=lambda k: reg_results[k]['r2'])
print(f"\n✓ Best Regressor: {best_reg_name} (R²={reg_results[best_reg_name]['r2']:.4f})")


In [ ]:
# ─── 5.2 REGRESSION VISUALISATION ────────────────────────────────────────────
best_reg = reg_results[best_reg_name]['model']
y_pred_best = reg_results[best_reg_name]['y_pred']

fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# Pred vs actual scatter
axes[0,0].scatter(y_test_r[:3000], y_pred_best[:3000],
                  alpha=0.25, s=12, color=PALETTE['sky'], edgecolors='none')
mn, mx = y_test_r.min(), y_test_r.max()
axes[0,0].plot([mn,mx],[mn,mx],'r--', lw=2, label='Perfect fit')
axes[0,0].set_xlabel('Actual Rating'); axes[0,0].set_ylabel('Predicted Rating')
axes[0,0].set_title(f'Predicted vs Actual — {best_reg_name}', fontsize=12, fontweight='bold')
axes[0,0].legend(fontsize=9)
r2_val = reg_results[best_reg_name]['r2']
axes[0,0].text(0.05, 0.92, f'R² = {r2_val:.4f}', transform=axes[0,0].transAxes,
               fontsize=11, fontweight='bold',
               bbox=dict(boxstyle='round', facecolor=PALETTE['sky_l'], alpha=0.8))

# Residuals
residuals = y_test_r - y_pred_best
axes[0,1].scatter(y_pred_best[:3000], residuals[:3000],
                  alpha=0.2, s=10, color=PALETTE['mint'], edgecolors='none')
axes[0,1].axhline(0, color='red', lw=1.5, ls='--')
axes[0,1].set_xlabel('Predicted Rating'); axes[0,1].set_ylabel('Residual')
axes[0,1].set_title('Residual Plot', fontsize=12, fontweight='bold')

# Residual histogram
axes[0,2].hist(residuals, bins=40, color=PALETTE['sky_l'], edgecolor=PALETTE['sky'], lw=0.6)
axes[0,2].axvline(residuals.mean(), color=PALETTE['sand'], lw=2, ls='--',
                  label=f'Mean={residuals.mean():.4f}')
axes[0,2].set_xlabel('Residual'); axes[0,2].set_ylabel('Count')
axes[0,2].set_title('Residual Distribution', fontsize=12, fontweight='bold')
axes[0,2].legend()

# Regressor R² comparison
r2_vals  = [reg_results[k]['r2'] for k in reg_results]
rmse_vals= [reg_results[k]['rmse'] for k in reg_results]
model_names_r = list(reg_results.keys())
colors_r = [PALETTE['sky'] if k == best_reg_name else PALETTE['sand'] for k in model_names_r]
axes[1,0].bar(model_names_r, r2_vals, color=colors_r, edgecolor='white')
for i, v in enumerate(r2_vals):
    axes[1,0].text(i, v+0.003, f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='600')
axes[1,0].set_title('Regressor R² Comparison', fontsize=12, fontweight='bold')
axes[1,0].set_xticklabels(model_names_r, rotation=20, ha='right', fontsize=9)
axes[1,0].set_ylabel('R² Score')

# RMSE comparison
axes[1,1].bar(model_names_r, rmse_vals, color=[PALETTE['mint'] if k==best_reg_name
              else PALETTE['sand'] for k in model_names_r], edgecolor='white')
for i, v in enumerate(rmse_vals):
    axes[1,1].text(i, v+0.001, f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='600')
axes[1,1].set_title('Regressor RMSE Comparison (lower=better)', fontsize=12, fontweight='bold')
axes[1,1].set_xticklabels(model_names_r, rotation=20, ha='right', fontsize=9)
axes[1,1].set_ylabel('RMSE')

# Feature importance
if hasattr(best_reg, 'feature_importances_'):
    fi_r = pd.Series(best_reg.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    axes[1,2].barh(range(len(fi_r)), fi_r.values[::-1],
                   color=plt.cm.Greens_r(np.linspace(0.2,0.85,len(fi_r)))[::-1])
    axes[1,2].set_yticks(range(len(fi_r)))
    axes[1,2].set_yticklabels(fi_r.index[::-1], fontsize=9)
    axes[1,2].set_title(f'Feature Importances — {best_reg_name}', fontsize=12, fontweight='bold')
    axes[1,2].set_xlabel('Importance')

plt.suptitle('Regression Analysis — Target: attraction rating', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Clustering — Traveler Segment Profiling

In [ ]:
# ─── 6.1 KMEANS CLUSTERING WITH PCA ──────────────────────────────────────────
# Cluster on attraction-level features from catalog
num_feats = ['avg_rating','avg_ticket_price','avg_spend','avg_duration',
             'popularity_score','visit_count','highly_recommended_pct','very_satisfied_pct']
num_feats = [f for f in num_feats if f in catalog.columns]

X_cat = catalog[num_feats].fillna(0).values
scaler_clust = StandardScaler()
X_scaled = scaler_clust.fit_transform(X_cat)

# Elbow + Silhouette method
k_range = range(2, 10)
inertias, silhouettes = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_range), inertias, 'o-', color=PALETTE['sky'], lw=2, markersize=8,
             markerfacecolor='white', markeredgewidth=2)
axes[0].fill_between(list(k_range), inertias, alpha=0.1, color=PALETTE['sky'])
axes[0].set_xlabel('Number of Clusters K'); axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')
axes[0].axvline(6, color=PALETTE['sand'], ls='--', lw=2, label='K=6 chosen')
axes[0].legend()

axes[1].plot(list(k_range), silhouettes, 's-', color=PALETTE['mint'], lw=2, markersize=8,
             markerfacecolor='white', markeredgewidth=2)
axes[1].fill_between(list(k_range), silhouettes, alpha=0.1, color=PALETTE['mint'])
best_k_sil = list(k_range)[np.argmax(silhouettes)]
axes[1].axvline(best_k_sil, color=PALETTE['sand'], ls='--', lw=2,
                label=f'Best silhouette K={best_k_sil}')
axes[1].set_xlabel('Number of Clusters K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].legend()

plt.suptitle('K Selection — Elbow + Silhouette Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Optimal K by silhouette : {best_k_sil}")
print(f"Chosen K for production : 6 (business interpretability)")


In [ ]:
# ─── 6.2 FIT FINAL CLUSTERING + PCA VISUALIZATION ────────────────────────────
K_FINAL = 6
kmeans_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10, max_iter=300)
cluster_labels = kmeans_final.fit_predict(X_scaled)
catalog_cl = catalog.copy()
catalog_cl['cluster'] = cluster_labels

sil_score = silhouette_score(X_scaled, cluster_labels)
print(f"KMeans K={K_FINAL} Silhouette Score: {sil_score:.4f}")

# PCA 2D projection
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA Explained Variance   : {pca.explained_variance_ratio_[0]*100:.1f}% + {pca.explained_variance_ratio_[1]*100:.1f}% = {sum(pca.explained_variance_ratio_)*100:.1f}%")

CLUSTER_NAMES = {
    0: 'Budget Cultural\nExplorers',
    1: 'Luxury Nature\nSeekers',
    2: 'Family Heritage\nTravelers',
    3: 'Adventure\nEnthusiasts',
    4: 'Urban Experience\nSeekers',
    5: 'Spiritual & Wellness\nTourists',
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# PCA scatter
for k in range(K_FINAL):
    mask = cluster_labels == k
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[CHART_COLORS[k]], label=f'C{k}: {CLUSTER_NAMES[k].replace(chr(10), " ")}',
                    alpha=0.65, s=60, edgecolors='white', linewidths=0.5)
# Mark centroids
centers_pca = pca.transform(kmeans_final.cluster_centers_)
axes[0].scatter(centers_pca[:,0], centers_pca[:,1],
                c='white', marker='*', s=300, zorder=5, edgecolors='black', linewidths=1.5)
axes[0].set_title(f'KMeans Clustering — PCA 2D Projection (K={K_FINAL})', fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
axes[0].legend(fontsize=8, loc='upper right')

# Cluster profile heatmap
cluster_profiles = catalog_cl.groupby('cluster')[num_feats].mean()
cluster_profiles_norm = (cluster_profiles - cluster_profiles.min()) / (cluster_profiles.max() - cluster_profiles.min() + 1e-9)
cluster_profiles_norm.index = [CLUSTER_NAMES[k].replace('\n',' ') for k in cluster_profiles_norm.index]
sns.heatmap(cluster_profiles_norm.T, annot=cluster_profiles.T.round(2),
            fmt='.2f', cmap='Blues', ax=axes[1], linewidths=0.4,
            cbar_kws={'label': 'Normalized Value', 'shrink': 0.7},
            annot_kws={'size': 8})
axes[1].set_title('Cluster Profile Heatmap (Normalized)', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=25, ha='right', fontsize=8)
axes[1].set_yticklabels(axes[1].get_yticklabels(), fontsize=9)

plt.tight_layout()
plt.savefig('../static/charts/nb2_clusters.png', bbox_inches='tight', dpi=130)
plt.show()


In [ ]:
# ─── 6.3 CLUSTER PROFILING TABLE ──────────────────────────────────────────────
print("=" * 80)
print(f"{'CLUSTER PROFILES':^80}")
print("=" * 80)
for k in range(K_FINAL):
    sub = catalog_cl[catalog_cl['cluster'] == k]
    top_cats = sub['category'].value_counts().head(3).to_dict()
    top_provs = sub['province'].value_counts().head(3).to_dict()
    top_budget = sub['budget_level'].value_counts().head(1).to_dict()

    print(f"\n  Cluster {k} — {CLUSTER_NAMES[k].replace(chr(10),' ')} ({len(sub)} attractions)")
    print(f"  {'─'*60}")
    print(f"  Avg Rating      : {sub['avg_rating'].mean():.3f}")
    print(f"  Avg Ticket Price: ¥{sub['avg_ticket_price'].mean():.0f}")
    print(f"  Avg Spend       : ¥{sub['avg_spend'].mean():.0f}")
    print(f"  Avg Duration    : {sub['avg_duration'].mean():.1f}h")
    print(f"  Avg Popularity  : {sub['popularity_score'].mean():.4f}")
    print(f"  Dominant Budget : {list(top_budget.keys())[0] if top_budget else 'N/A'}")
    print(f"  Top Categories  : {list(top_cats.keys())}")
    print(f"  Top Provinces   : {list(top_provs.keys())}")
    top5 = sub.nlargest(5,'popularity_score')['attraction_name'].tolist()
    print(f"  Top 5 Attractions: {top5}")
print("=" * 80)


## 7. Hybrid Recommender — Production System

In [ ]:
# ─── 7.1 HYBRID ARCHITECTURE ──────────────────────────────────────────────────
from src.recommender import HybridRecommender

HYBRID_WEIGHTS = {
    'similarity':  0.30,   # TF-IDF cosine similarity to user query
    'popularity':  0.25,   # Popularity score (freq × rating × rec%)
    'rating':      0.20,   # Normalized average rating
    'budget':      0.10,   # Budget level compatibility
    'season':      0.10,   # Seasonal match bonus
    'category':    0.05,   # Category alignment bonus
}

print("Hybrid Recommender Weight Configuration:")
print("─" * 45)
total = sum(HYBRID_WEIGHTS.values())
for component, weight in HYBRID_WEIGHTS.items():
    bar = '█' * int(weight * 100 // 4)
    print(f"  {component:<15} {weight:.2f}  {bar}")
print(f"{'─'*45}")
print(f"  Total            {total:.2f}")

hybrid = HybridRecommender(weights=HYBRID_WEIGHTS)
hybrid.fit(catalog, content_rec)
print("\n✓ Hybrid Recommender ready")


In [ ]:
# ─── 7.2 COMPREHENSIVE USE CASE TESTS ────────────────────────────────────────
use_cases = [
    {'label': 'Case 1: Budget Spring Nature Seeker',
     'params': {'category':'Natural Scenery','season':'Spring','budget_level':'Budget','min_rating':4.0,'n':6}},
    {'label': 'Case 2: Autumn Cultural History Enthusiast (Premium)',
     'params': {'category':'Historical Culture','season':'Autumn','budget_level':'Premium','min_rating':4.2,'n':6}},
    {'label': 'Case 3: Winter Religious Site Explorer',
     'params': {'category':'Religious Culture','season':'Winter','min_rating':4.0,'n':6}},
    {'label': 'Case 4: Find similar to Xi Hu (destination-to-destination)',
     'params': {'attraction_name':'Xi Hu','n':6}},
    {'label': 'Case 5: Summer Theme Park Family Fun',
     'params': {'category':'Theme Park','season':'Summer','min_rating':3.8,'n':6}},
    {'label': 'Case 6: Ancient Town Budget Traveler',
     'params': {'category':'Ancient Town','budget_level':'Budget','min_rating':4.0,'n':6}},
]

for uc in use_cases:
    print("═" * 70)
    print(f"  {uc['label']}")
    print("═" * 70)
    result = hybrid.recommend(**uc['params'])
    display(result[['attraction_name','province','category','avg_rating',
                     'budget_level','best_season','hybrid_score','match_reasons']]
        .style.background_gradient(subset=['hybrid_score'], cmap='Blues')
        .format({'avg_rating':'{:.3f}','hybrid_score':'{:.4f}'})
        .set_properties(**{'font-size':'10.5px'}))
    print()


In [ ]:
# ─── 7.3 HYBRID vs BASELINE COMPARISON ───────────────────────────────────────
# Ablation: compare hybrid vs pure popularity vs pure content for same query
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

test_params = {'category':'Natural Scenery', 'season':'Autumn', 'min_rating':4.0, 'n':10}

# Pure popularity
pop_result = pop_rec.recommend(category='Natural Scenery', season='Autumn', n=10)

# Pure content
ct_result  = content_rec.search_by_query('Natural Scenery Autumn China', n=10)
ct_result  = ct_result[ct_result['avg_rating'] >= 4.0]

# Hybrid
hy_result  = hybrid.recommend(**test_params)

systems = [
    ('Popularity Baseline', pop_result, 'popularity_score', PALETTE['sand']),
    ('Content-Based', ct_result, 'similarity_score', PALETTE['mint']),
    ('Hybrid Recommender', hy_result, 'hybrid_score', PALETTE['sky']),
]

for ax, (name, result, score_col, color) in zip(axes, systems):
    if score_col not in result.columns:
        result = result.copy()
        result[score_col] = result['popularity_score'] if 'popularity_score' in result.columns else 0.5
    n_show = min(10, len(result))
    r = result.head(n_show)
    y_vals = range(n_show)
    ax.barh(list(y_vals), r[score_col].values[::-1],
            color=color, alpha=0.85, edgecolor='white')
    ax.set_yticks(list(y_vals))
    ax.set_yticklabels(r['attraction_name'].values[::-1], fontsize=8)
    ax.set_xlabel('Score')
    ax.set_title(f'{name}\n(avg_rating: {r["avg_rating"].mean():.3f})',
                 fontsize=11, fontweight='bold')
    for i, (v, rating) in enumerate(zip(r[score_col].values[::-1], r['avg_rating'].values[::-1])):
        ax.text(v + 0.005, i, f'★{rating:.1f}', va='center', fontsize=8)

plt.suptitle('System Comparison: Popularity vs Content vs Hybrid\n(Natural Scenery, Autumn, min_rating≥4.0)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/charts/nb2_system_comparison.png', bbox_inches='tight', dpi=130)
plt.show()

# Coverage analysis
print("Coverage Analysis:")
print(f"  Popularity Baseline  — Unique categories: {pop_result['category'].nunique()}")
print(f"  Content-Based        — Unique categories: {ct_result['category'].nunique() if not ct_result.empty else 0}")
print(f"  Hybrid Recommender   — Unique categories: {hy_result['category'].nunique()}")
print(f"\nAvg Rating in Results:")
print(f"  Popularity Baseline  — {pop_result['avg_rating'].mean():.4f}")
print(f"  Content-Based        — {ct_result['avg_rating'].mean():.4f}" if not ct_result.empty else "")
print(f"  Hybrid Recommender   — {hy_result['avg_rating'].mean():.4f}")


In [ ]:
# ─── 7.4 RECOMMENDATION EVALUATION METRICS ───────────────────────────────────
print("=" * 65)
print("RECOMMENDATION SYSTEM EVALUATION")
print("=" * 65)

# Coverage: % of catalog surfaced in top-10 recommendations
n_test_queries = 50
all_recs = set()
test_categories = catalog['category'].value_counts().head(10).index.tolist()
for cat in test_categories:
    for season in ['Spring','Summer','Autumn','Winter']:
        try:
            r = hybrid.recommend(category=cat, season=season, n=10)
            all_recs.update(r['attraction_name'].tolist())
        except:
            pass

coverage = len(all_recs) / len(catalog)
print(f"\n  Coverage (across {n_test_queries} diverse queries):")
print(f"    Attractions surfaced : {len(all_recs)} / {len(catalog)}")
print(f"    Coverage %           : {coverage*100:.1f}%")

# Diversity: average pairwise category diversity
def diversity_score(recs):
    cats = recs['category'].tolist()
    if len(cats) < 2: return 0.0
    unique_pairs = len(set(cats))
    return unique_pairs / len(cats)

div_scores = []
for cat in test_categories[:5]:
    r = hybrid.recommend(category=cat, n=10)
    div_scores.append(diversity_score(r))
print(f"\n  Diversity (avg category variety in top-10):")
print(f"    Average diversity    : {np.mean(div_scores):.4f}")

# Novelty: fraction of results that are NOT top-20% popular
top_pop = catalog.nlargest(int(len(catalog)*0.20), 'popularity_score')['attraction_name'].tolist()
novelty_scores = []
for cat in test_categories[:5]:
    r = hybrid.recommend(category=cat, n=10)
    novel = (~r['attraction_name'].isin(top_pop)).mean()
    novelty_scores.append(novel)
print(f"\n  Novelty (fraction non-top-20% popular in results):")
print(f"    Average novelty      : {np.mean(novelty_scores):.4f}")

# Avg quality
quality_scores = []
for cat in test_categories:
    r = hybrid.recommend(category=cat, n=10, min_rating=4.0)
    quality_scores.append(r['avg_rating'].mean())
print(f"\n  Quality (avg rating across recommendation sets):")
print(f"    Average quality      : {np.mean(quality_scores):.4f}")

print("=" * 65)
print("\n✓ All 7 recommendation approaches complete")
print("  → Run Notebook 03 for hyperparameter tuning")
